# 45｜从零实现 ELECTRA：小 Generator、替换采样与 Replaced-Token Detection

本 Notebook 不调用 transformers、nn.Transformer 或 nn.MultiheadAttention。我们手写双向 encoder attention、generator MLM、从词表分布采样 replacement、discriminator 的 replaced-token detection（RTD）以及联合 loss。

ELECTRA 的关键边界是：被选为 MLM 的位置不一定真的被替换。若 generator 恰好采样回原 token，RTD 标签必须是 0；标签应由 original_ids 与 corrupted_ids 的逐位置比较得到。special token 和 padding 不参加 MLM，也不参加本教学版 RTD loss。

> 实验边界：CPU、离线、单线程、小合成数据；受控 loss 下降只验证预训练数据流，不能代表下游迁移或真实语料泛化。

## 1. ELECTRA 数据流与张量合同

1. 从每行普通 token 中选择 MLM 位置，把输入改成 MASK，未选 label 设为 -100。
2. generator 输出 [B,T,V]，只在被选位置计算 token CE。
3. 从 generator 的普通词分布采样 replacement，并写回原序列。
4. discriminator 读取 corrupted sequence，逐 token 输出一个 logit。
5. RTD 标签是 corrupted!=original；special/padding 的标签为 -100 并排除 BCE。

attention 隐状态为 [B,T,D]、拆头 [B,H,T,d_h]、权重 [B,H,T,T]。双向 encoder 没有 causal mask，复杂度主项为 O(B·H·T²)。

In [ ]:
import copy  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import warnings  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。

warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED45 = 4507  # 计算并保存当前步骤的中间状态。
random.seed(SEED45)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED45)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE45 = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

PAD45, CLS45, SEP45, MASK45, UNK45 = 0, 1, 2, 3, 4  # 计算并保存当前步骤的中间状态。
VOCAB45 = [  # 计算并保存当前步骤的中间状态。
    "<pad>", "<cls>", "<sep>", "<mask>", "<unk>",  # 执行当前语句以推进本节示例。
    "北京", "上海", "天气", "晴朗", "今天", "明天", "模型", "学习",  # 执行当前语句以推进本节示例。
    "文本", "图像", "检索", "知识", "系统", "数据", "分类", "生成",  # 执行当前语句以推进本节示例。
    "语言", "理解", "搜索",  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
TOKEN_TO_ID45 = {token: index for index, token in enumerate(VOCAB45)}  # 计算并保存当前步骤的中间状态。
SPECIAL45 = {PAD45, CLS45, SEP45, MASK45, UNK45}  # 计算并保存当前步骤的中间状态。
ORDINARY45 = list(range(max(SPECIAL45) + 1, len(VOCAB45)))  # 计算并保存当前步骤的中间状态。

assert DEVICE45.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert len(VOCAB45) == len(TOKEN_TO_ID45)  # 用受控断言验证关键不变量。
assert set(ORDINARY45).isdisjoint(SPECIAL45)  # 用受控断言验证关键不变量。
assert ORDINARY45[0] == 5 and ORDINARY45[-1] == len(VOCAB45) - 1  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE45), "vocab": len(VOCAB45)})  # 执行当前语句以推进本节示例。

## 2. 先切分原始记录，再选择 MLM 位置

数据切分的单位应是原始文档或语义组，而不是 masking 后的样本。若先生成多个 mask 版本再随机切分，同一原句很可能同时出现在 train/validation，导致验证泄漏。

下面每行使用局部 Generator 选择普通 token；只要存在候选且 probability>0，就保证至少一个监督位置。局部 RNG 让同 seed 可重现且不污染全局随机状态。

selected 位置采用显式 **80/10/10** generator 输入策略：80% 写 MASK、10% 写随机普通词、10% 保持原词。位置选择、策略动作和随机普通词分别使用由基础 seed 派生的三个独立 `torch.Generator`；随机词即使碰巧等于原词，策略仍记录为 random。`policy` 用 `0/1/2` 表示三种动作，未选择位置为 -1，便于直接审计而不是从最终 token 猜动作。


In [ ]:
RAW_RECORDS45 = [  # 计算并保存当前步骤的中间状态。
    {"id": "e0", "tokens": [5, 9, 7, 8, 17]},  # 执行当前语句以推进本节示例。
    {"id": "e1", "tokens": [6, 10, 7, 8, 17]},  # 执行当前语句以推进本节示例。
    {"id": "e2", "tokens": [11, 12, 13, 23, 17]},  # 执行当前语句以推进本节示例。
    {"id": "e3", "tokens": [14, 15, 18, 16, 17]},  # 执行当前语句以推进本节示例。
    {"id": "e4", "tokens": [19, 13, 11, 12, 20]},  # 执行当前语句以推进本节示例。
    {"id": "e5", "tokens": [21, 22, 13, 20, 23]},  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
TRAIN_IDS45, VALID_IDS45 = ["e0", "e1", "e2", "e3"], ["e4", "e5"]  # 计算并保存当前步骤的中间状态。
POLICY_SEED_OFFSET45 = 1_000_003  # 计算并保存当前步骤的中间状态。
RANDOM_TOKEN_SEED_OFFSET45 = 2_000_003  # 计算并保存当前步骤的中间状态。
REPLACEMENT_SEED_OFFSET45 = 10_000  # 计算并保存当前步骤的中间状态。


def collate_records45(records):  # 定义本节可复用的核心函数。
    sequences = [[CLS45, *record["tokens"], SEP45] for record in records]  # 计算并保存当前步骤的中间状态。
    max_length = max(len(sequence) for sequence in sequences)  # 计算并保存当前步骤的中间状态。
    ids = torch.full((len(sequences), max_length), PAD45, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    mask = torch.zeros_like(ids, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    for row, sequence in enumerate(sequences):  # 遍历输入元素以累积或检查结果。
        ids[row, :len(sequence)] = torch.tensor(sequence)  # 计算并保存当前步骤的中间状态。
        mask[row, :len(sequence)] = True  # 计算并保存当前步骤的中间状态。
    return ids, mask  # 返回当前分支计算出的结果。


def choose_mlm45(input_ids, attention_mask, probability, seed):  # 定义本节可复用的核心函数。
    if input_ids.dtype != torch.long or input_ids.ndim != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("input_ids 必须是二维 long")  # 遇到非法合同立即显式失败。
    if attention_mask.shape != input_ids.shape or attention_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
        raise ValueError("attention_mask 合同错误")  # 遇到非法合同立即显式失败。
    if not 0.0 <= probability <= 1.0:  # 按当前条件选择后续控制路径。
        raise ValueError("probability 必须在 [0,1]")  # 遇到非法合同立即显式失败。
    eligible = attention_mask.clone()  # 计算并保存当前步骤的中间状态。
    for special in SPECIAL45:  # 遍历输入元素以累积或检查结果。
        eligible &= input_ids.ne(special)  # 计算并保存当前步骤的中间状态。
    selected = torch.zeros_like(eligible)  # 计算并保存当前步骤的中间状态。
    selection_generator = torch.Generator().manual_seed(seed)  # 计算并保存当前步骤的中间状态。
    for row in range(input_ids.shape[0]):  # 遍历输入元素以累积或检查结果。
        candidates = eligible[row].nonzero(as_tuple=False).flatten()  # 计算并保存当前步骤的中间状态。
        if candidates.numel() and probability > 0:  # 按当前条件选择后续控制路径。
            count = max(1, int(round(candidates.numel() * probability)))  # 计算并保存当前步骤的中间状态。
            permutation = torch.randperm(candidates.numel(), generator=selection_generator)  # 计算并保存当前步骤的中间状态。
            selected[row, candidates[permutation[:count]]] = True  # 计算并保存当前步骤的中间状态。

    labels = input_ids.masked_fill(~selected, -100)  # 计算并保存当前步骤的中间状态。
    generator_input = input_ids.clone()  # 计算并保存当前步骤的中间状态。
    policy = torch.full_like(input_ids, -1)  # -1未选；0 MASK；1随机普通词；2保持原词
    selected_positions = selected.nonzero(as_tuple=False)  # 计算并保存当前步骤的中间状态。
    if selected_positions.numel():  # 按当前条件选择后续控制路径。
        policy_generator = torch.Generator().manual_seed(seed + POLICY_SEED_OFFSET45)  # 计算并保存当前步骤的中间状态。
        random_token_generator = torch.Generator().manual_seed(seed + RANDOM_TOKEN_SEED_OFFSET45)  # 计算并保存当前步骤的中间状态。
        draws = torch.rand(selected_positions.shape[0], generator=policy_generator)  # 计算并保存当前步骤的中间状态。
        actions = torch.where(draws < 0.8, 0, torch.where(draws < 0.9, 1, 2)).long()  # 计算并保存当前步骤的中间状态。
        policy[selected] = actions  # 计算并保存当前步骤的中间状态。
        mask_positions = selected_positions[actions == 0]  # 计算并保存当前步骤的中间状态。
        random_positions = selected_positions[actions == 1]  # 计算并保存当前步骤的中间状态。
        if mask_positions.numel():  # 按当前条件选择后续控制路径。
            generator_input[mask_positions[:, 0], mask_positions[:, 1]] = MASK45  # 计算并保存当前步骤的中间状态。
        if random_positions.numel():  # 按当前条件选择后续控制路径。
            sampled = torch.randint(  # 计算并保存当前步骤的中间状态。
                0, len(ORDINARY45), (random_positions.shape[0],), generator=random_token_generator  # 计算并保存当前步骤的中间状态。
            )  # 执行当前语句以推进本节示例。
            ordinary = torch.tensor(ORDINARY45)  # 计算并保存当前步骤的中间状态。
            generator_input[random_positions[:, 0], random_positions[:, 1]] = ordinary[sampled]  # 计算并保存当前步骤的中间状态。
    return generator_input, labels, selected, policy  # 返回当前分支计算出的结果。


train_records45 = [record for record in RAW_RECORDS45 if record["id"] in TRAIN_IDS45]  # 计算并保存当前步骤的中间状态。
train_ids45, train_attention45 = collate_records45(train_records45)  # 计算并保存当前步骤的中间状态。
generator_input45, mlm_labels45, selected45, policy45 = choose_mlm45(  # 计算并保存当前步骤的中间状态。
    train_ids45, train_attention45, probability=0.4, seed=SEED45  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
generator_input_again45, labels_again45, selected_again45, policy_again45 = choose_mlm45(  # 计算并保存当前步骤的中间状态。
    train_ids45, train_attention45, probability=0.4, seed=SEED45  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。

assert generator_input45.equal(generator_input_again45)  # 用受控断言验证关键不变量。
assert mlm_labels45.equal(labels_again45) and selected45.equal(selected_again45)  # 用受控断言验证关键不变量。
assert policy45.equal(policy_again45)  # 用受控断言验证关键不变量。
assert selected45.sum(dim=1).eq(2).all()  # 用受控断言验证关键不变量。
assert policy45[~selected45].eq(-1).all()  # 用受控断言验证关键不变量。
assert set(policy45[selected45].tolist()).issubset({0, 1, 2})  # 用受控断言验证关键不变量。
assert generator_input45[(policy45 == 0)].eq(MASK45).all()  # 用受控断言验证关键不变量。
assert mlm_labels45[~selected45].eq(-100).all()  # 用受控断言验证关键不变量。
for special45 in SPECIAL45:  # 遍历输入元素以累积或检查结果。
    assert not bool((selected45 & train_ids45.eq(special45)).any())  # 用受控断言验证关键不变量。

# 大样本只检验策略概率，不把“随机词碰巧等于原词”误算成 unchanged action。
policy_probe_ids45 = torch.tensor(ORDINARY45 * 100).reshape(100, len(ORDINARY45))  # 计算并保存当前步骤的中间状态。
policy_probe_mask45 = torch.ones_like(policy_probe_ids45, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
_, _, policy_probe_selected45, policy_probe45 = choose_mlm45(  # 计算并保存当前步骤的中间状态。
    policy_probe_ids45, policy_probe_mask45, probability=1.0, seed=SEED45 + 333  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
policy_counts45 = torch.bincount(policy_probe45[policy_probe_selected45], minlength=3).float()  # 计算并保存当前步骤的中间状态。
policy_rates45 = policy_counts45 / policy_counts45.sum()  # 计算并保存当前步骤的中间状态。
assert 0.76 < float(policy_rates45[0]) < 0.84  # 用受控断言验证关键不变量。
assert 0.07 < float(policy_rates45[1]) < 0.13  # 用受控断言验证关键不变量。
assert 0.07 < float(policy_rates45[2]) < 0.13  # 用受控断言验证关键不变量。
assert set(generator_input45[(policy45 == 1)].tolist()).issubset(set(ORDINARY45))  # 用受控断言验证关键不变量。
assert set(TRAIN_IDS45).isdisjoint(VALID_IDS45)  # 用受控断言验证关键不变量。
assert set(TRAIN_IDS45) | set(VALID_IDS45) == {record["id"] for record in RAW_RECORDS45}  # 用受控断言验证关键不变量。

## 3. 手写双向 self-attention

Q、K、V 线性投影后拆头，score=QKᵀ/sqrt(d_h)。这里只屏蔽 padding key，没有 causal 上三角，因此右侧 token 可以影响左侧表示。softmax 后把 padding query 的权重与输出显式归零，避免 output projection bias 重新引入非零值。

In [ ]:
class ManualSelfAttention45(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, heads):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if dim % heads:  # 按当前条件选择后续控制路径。
            raise ValueError("dim 必须整除 heads")  # 遇到非法合同立即显式失败。
        self.dim, self.heads, self.head_dim = dim, heads, dim // heads  # 计算并保存当前步骤的中间状态。
        self.q_proj = nn.Linear(dim, dim)  # 计算并保存当前步骤的中间状态。
        self.k_proj = nn.Linear(dim, dim)  # 计算并保存当前步骤的中间状态。
        self.v_proj = nn.Linear(dim, dim)  # 计算并保存当前步骤的中间状态。
        self.out_proj = nn.Linear(dim, dim)  # 计算并保存当前步骤的中间状态。

    def _split(self, x):  # 定义本节可复用的核心函数。
        batch, length, _ = x.shape  # 计算并保存当前步骤的中间状态。
        return x.view(batch, length, self.heads, self.head_dim).transpose(1, 2)  # 返回当前分支计算出的结果。

    def forward(self, x, attention_mask):  # 定义本节可复用的核心函数。
        if x.ndim != 3 or x.shape[-1] != self.dim:  # 按当前条件选择后续控制路径。
            raise ValueError("attention 输入形状错误")  # 遇到非法合同立即显式失败。
        if attention_mask.shape != x.shape[:2] or attention_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("attention_mask 合同错误")  # 遇到非法合同立即显式失败。
        if not bool(attention_mask.any(dim=1).all()):  # 按当前条件选择后续控制路径。
            raise ValueError("每行至少需要一个有效 token")  # 遇到非法合同立即显式失败。
        q, k, v = self._split(self.q_proj(x)), self._split(self.k_proj(x)), self._split(self.v_proj(x))  # 计算并保存当前步骤的中间状态。
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # 计算并保存当前步骤的中间状态。
        visible = attention_mask[:, None, None, :]  # 计算并保存当前步骤的中间状态。
        weights = torch.softmax(scores.masked_fill(~visible, -torch.inf), dim=-1)  # 计算并保存当前步骤的中间状态。
        weights = weights * attention_mask[:, None, :, None]  # 计算并保存当前步骤的中间状态。
        context = torch.matmul(weights, v).transpose(1, 2).contiguous().view_as(x)  # 计算并保存当前步骤的中间状态。
        output = self.out_proj(context) * attention_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return output, weights  # 返回当前分支计算出的结果。


attention_probe45 = ManualSelfAttention45(4, 2)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    for layer45 in [  # 遍历输入元素以累积或检查结果。
        attention_probe45.q_proj, attention_probe45.k_proj,  # 执行当前语句以推进本节示例。
        attention_probe45.v_proj, attention_probe45.out_proj,  # 执行当前语句以推进本节示例。
    ]:  # 执行当前语句以推进本节示例。
        layer45.weight.copy_(torch.eye(4))  # 执行当前语句以推进本节示例。
        layer45.bias.zero_()  # 执行当前语句以推进本节示例。
x_probe45 = torch.tensor([[[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]]])  # 计算并保存当前步骤的中间状态。
mask_probe45 = torch.ones(1, 2, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
output_probe45, weights_probe45 = attention_probe45(x_probe45, mask_probe45)  # 计算并保存当前步骤的中间状态。
expected_scores45 = torch.tensor([[1.0, 0.0], [0.0, 1.0]]) / math.sqrt(2.0)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(weights_probe45[0, 0], torch.softmax(expected_scores45, -1), atol=1e-7)  # 用受控断言验证关键不变量。
assert torch.allclose(weights_probe45[0, 1], torch.full((2, 2), 0.5), atol=1e-7)  # 用受控断言验证关键不变量。

padded_mask45 = torch.tensor([[True, False]])  # 计算并保存当前步骤的中间状态。
padded_output45, padded_weights45 = attention_probe45(x_probe45, padded_mask45)  # 计算并保存当前步骤的中间状态。
assert padded_output45[0, 1].abs().max().item() == 0.0  # 用受控断言验证关键不变量。
assert padded_weights45[0, :, 1].abs().max().item() == 0.0  # 用受控断言验证关键不变量。
assert padded_weights45[0, :, 0, 1].abs().max().item() == 0.0  # 用受控断言验证关键不变量。

## 4. Generator 与 Discriminator 的容量和共享策略

论文中 generator 通常比 discriminator 小，但可以共享 token embedding。为让教学代码简洁，两者 hidden dim 相同、generator 一层、discriminator 两层，并共享同一个 embedding Parameter；encoder block、position embedding 与 attention 参数各自独立。

共享能减少参数并让两种目标共同更新词表示，但也耦合了优化。生产实验应把共享开关、容量比和 loss 权重作为显式配置，不能只在代码中默默决定。

In [ ]:
class EncoderBlock45(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, heads, hidden_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.norm1 = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.attention = ManualSelfAttention45(dim, heads)  # 计算并保存当前步骤的中间状态。
        self.norm2 = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.ff1 = nn.Linear(dim, hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.ff2 = nn.Linear(hidden_dim, dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, x, attention_mask):  # 定义本节可复用的核心函数。
        update, _ = self.attention(self.norm1(x), attention_mask)  # 计算并保存当前步骤的中间状态。
        x = (x + update) * attention_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        update = self.ff2(F.gelu(self.ff1(self.norm2(x))))  # 计算并保存当前步骤的中间状态。
        return (x + update) * attention_mask.unsqueeze(-1)  # 返回当前分支计算出的结果。


class TinyEncoder45(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size, dim, heads, hidden_dim, layers, max_length, embedding=None):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.token_embedding = embedding if embedding is not None else nn.Embedding(  # 计算并保存当前步骤的中间状态。
            vocab_size, dim, padding_idx=PAD45  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
        if self.token_embedding.embedding_dim != dim:  # 按当前条件选择后续控制路径。
            raise ValueError("共享 embedding dim 不匹配")  # 遇到非法合同立即显式失败。
        self.position_embedding = nn.Embedding(max_length, dim)  # 计算并保存当前步骤的中间状态。
        self.blocks = nn.ModuleList(  # 计算并保存当前步骤的中间状态。
            [EncoderBlock45(dim, heads, hidden_dim) for _ in range(layers)]  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        self.final_norm = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.max_length = max_length  # 计算并保存当前步骤的中间状态。

    def forward(self, input_ids, attention_mask):  # 定义本节可复用的核心函数。
        if input_ids.dtype != torch.long or input_ids.ndim != 2:  # 按当前条件选择后续控制路径。
            raise ValueError("input_ids 必须是二维 long")  # 遇到非法合同立即显式失败。
        if attention_mask.shape != input_ids.shape or attention_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("attention_mask 合同错误")  # 遇到非法合同立即显式失败。
        if input_ids.shape[1] > self.max_length:  # 按当前条件选择后续控制路径。
            raise ValueError("序列超过 position embedding 上限")  # 遇到非法合同立即显式失败。
        if not bool(attention_mask.any(dim=1).all()):  # 按当前条件选择后续控制路径。
            raise ValueError("每行至少一个有效 token")  # 遇到非法合同立即显式失败。
        positions = torch.arange(input_ids.shape[1], device=input_ids.device)  # 计算并保存当前步骤的中间状态。
        hidden = self.token_embedding(input_ids) + self.position_embedding(positions)[None]  # 计算并保存当前步骤的中间状态。
        hidden = hidden * attention_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        for block in self.blocks:  # 遍历输入元素以累积或检查结果。
            hidden = block(hidden, attention_mask)  # 计算并保存当前步骤的中间状态。
        return self.final_norm(hidden) * attention_mask.unsqueeze(-1)  # 返回当前分支计算出的结果。


class ElectraPretrainer45(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(  # 定义本节可复用的核心函数。
        self, vocab_size, dim=16, heads=4, hidden_dim=32,  # 计算并保存当前步骤的中间状态。
        generator_layers=1, discriminator_layers=2, max_length=16,  # 计算并保存当前步骤的中间状态。
        share_embeddings=True,  # 计算并保存当前步骤的中间状态。
    ):  # 执行当前语句以推进本节示例。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.config = {  # 计算并保存当前步骤的中间状态。
            "vocab_size": vocab_size, "dim": dim, "heads": heads,  # 执行当前语句以推进本节示例。
            "hidden_dim": hidden_dim, "generator_layers": generator_layers,  # 执行当前语句以推进本节示例。
            "discriminator_layers": discriminator_layers,  # 执行当前语句以推进本节示例。
            "max_length": max_length, "share_embeddings": share_embeddings,  # 执行当前语句以推进本节示例。
        }  # 执行当前语句以推进本节示例。
        shared = nn.Embedding(vocab_size, dim, padding_idx=PAD45) if share_embeddings else None  # 计算并保存当前步骤的中间状态。
        self.generator = TinyEncoder45(  # 计算并保存当前步骤的中间状态。
            vocab_size, dim, heads, hidden_dim, generator_layers, max_length, shared  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        discriminator_embedding = shared if share_embeddings else None  # 计算并保存当前步骤的中间状态。
        self.discriminator = TinyEncoder45(  # 计算并保存当前步骤的中间状态。
            vocab_size, dim, heads, hidden_dim, discriminator_layers,  # 执行当前语句以推进本节示例。
            max_length, discriminator_embedding,  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        self.generator_bias = nn.Parameter(torch.zeros(vocab_size))  # 计算并保存当前步骤的中间状态。
        self.discriminator_head = nn.Linear(dim, 1)  # 计算并保存当前步骤的中间状态。

    def generator_logits(self, masked_ids, attention_mask):  # 定义本节可复用的核心函数。
        hidden = self.generator(masked_ids, attention_mask)  # 计算并保存当前步骤的中间状态。
        return F.linear(hidden, self.generator.token_embedding.weight, self.generator_bias)  # 返回当前分支计算出的结果。

    def discriminator_logits(self, corrupted_ids, attention_mask):  # 定义本节可复用的核心函数。
        hidden = self.discriminator(corrupted_ids, attention_mask)  # 计算并保存当前步骤的中间状态。
        return self.discriminator_head(hidden).squeeze(-1)  # 返回当前分支计算出的结果。

    def forward(self, input_ids, attention_mask, head):  # 定义本节可复用的核心函数。
        if head == "generator":  # 按当前条件选择后续控制路径。
            return self.generator_logits(input_ids, attention_mask)  # 返回当前分支计算出的结果。
        if head == "discriminator":  # 按当前条件选择后续控制路径。
            return self.discriminator_logits(input_ids, attention_mask)  # 返回当前分支计算出的结果。
        raise ValueError("head 必须是 generator 或 discriminator")  # 遇到非法合同立即显式失败。


torch.manual_seed(SEED45)  # 执行当前语句以推进本节示例。
model45 = ElectraPretrainer45(len(VOCAB45))  # 计算并保存当前步骤的中间状态。
generator_logits_probe45 = model45.generator_logits(generator_input45, train_attention45)  # 计算并保存当前步骤的中间状态。
discriminator_logits_probe45 = model45.discriminator_logits(train_ids45, train_attention45)  # 计算并保存当前步骤的中间状态。
assert generator_logits_probe45.shape == (*train_ids45.shape, len(VOCAB45))  # 用受控断言验证关键不变量。
assert discriminator_logits_probe45.shape == train_ids45.shape  # 用受控断言验证关键不变量。
assert model45.generator.token_embedding.weight is model45.discriminator.token_embedding.weight  # 用受控断言验证关键不变量。
assert model45.generator.token_embedding.weight.data_ptr() == model45.discriminator.token_embedding.weight.data_ptr()  # 用受控断言验证关键不变量。

separate_model45 = ElectraPretrainer45(len(VOCAB45), share_embeddings=False)  # 计算并保存当前步骤的中间状态。
assert separate_model45.generator.token_embedding.weight is not separate_model45.discriminator.token_embedding.weight  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in model45.generator.parameters()) < sum(  # 用受控断言验证关键不变量。
    p.numel() for p in model45.discriminator.parameters()  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。

## 5. 从 generator 分布采样，而不是把 selected 直接当作 replaced

只在 selected 位置采样，并禁止生成 PAD/CLS/SEP/MASK/UNK。corrupted sequence 从 original 复制，再写入 sampled token；RTD truth 随后统一通过 corrupted!=original 计算。

若采样恰好等于原 token，该位置虽然产生 generator loss，但 discriminator 标签为 0。这是 ELECTRA 数据流中最重要的边界条件之一。

In [ ]:
def sample_replacements45(generator_logits, original_ids, selected, seed, temperature=1.0):  # 定义本节可复用的核心函数。
    if generator_logits.shape[:2] != original_ids.shape or selected.shape != original_ids.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("replacement 形状合同错误")  # 遇到非法合同立即显式失败。
    if selected.dtype != torch.bool or temperature <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("selected/temperature 合同错误")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(generator_logits).all():  # 按当前条件选择后续控制路径。
        raise ValueError("generator logits 必须有限")  # 遇到非法合同立即显式失败。
    corrupted = original_ids.clone()  # 计算并保存当前步骤的中间状态。
    local_generator = torch.Generator().manual_seed(seed)  # 计算并保存当前步骤的中间状态。
    ordinary_index = torch.tensor(ORDINARY45, device=generator_logits.device)  # 计算并保存当前步骤的中间状态。
    for row, position in selected.nonzero(as_tuple=False).tolist():  # 遍历输入元素以累积或检查结果。
        logits = generator_logits[row, position, ordinary_index] / temperature  # 计算并保存当前步骤的中间状态。
        probabilities = torch.softmax(logits, dim=-1)  # 计算并保存当前步骤的中间状态。
        sampled_local = torch.multinomial(probabilities.cpu(), 1, generator=local_generator).item()  # 计算并保存当前步骤的中间状态。
        corrupted[row, position] = ordinary_index[sampled_local]  # 计算并保存当前步骤的中间状态。
    return corrupted  # 返回当前分支计算出的结果。


def rtd_labels45(original_ids, corrupted_ids, attention_mask):  # 定义本节可复用的核心函数。
    if original_ids.shape != corrupted_ids.shape or original_ids.shape != attention_mask.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("RTD 输入形状错误")  # 遇到非法合同立即显式失败。
    eligible = attention_mask.clone()  # 计算并保存当前步骤的中间状态。
    for special in SPECIAL45:  # 遍历输入元素以累积或检查结果。
        eligible &= original_ids.ne(special)  # 计算并保存当前步骤的中间状态。
    labels = corrupted_ids.ne(original_ids).long()  # 计算并保存当前步骤的中间状态。
    return labels.masked_fill(~eligible, -100), eligible  # 返回当前分支计算出的结果。


selected_edge45 = torch.zeros(1, 4, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
selected_edge45[0, 1] = True  # 计算并保存当前步骤的中间状态。
original_edge45 = torch.tensor([[CLS45, 7, 8, SEP45]])  # 计算并保存当前步骤的中间状态。
same_logits45 = torch.full((1, 4, len(VOCAB45)), -100.0)  # 计算并保存当前步骤的中间状态。
same_logits45[0, 1, 7] = 100.0  # 计算并保存当前步骤的中间状态。
same_corrupted45 = sample_replacements45(  # 计算并保存当前步骤的中间状态。
    same_logits45, original_edge45, selected_edge45, seed=1  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
same_labels45, same_eligible45 = rtd_labels45(  # 计算并保存当前步骤的中间状态。
    original_edge45, same_corrupted45, torch.ones_like(original_edge45, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert same_corrupted45[0, 1].item() == original_edge45[0, 1].item()  # 用受控断言验证关键不变量。
assert same_labels45[0, 1].item() == 0  # 用受控断言验证关键不变量。
assert selected_edge45[0, 1] and same_labels45[0, 1].item() != 1  # 用受控断言验证关键不变量。

different_logits45 = torch.full((1, 4, len(VOCAB45)), -100.0)  # 计算并保存当前步骤的中间状态。
different_logits45[0, 1, 9] = 100.0  # 计算并保存当前步骤的中间状态。
different_corrupted45 = sample_replacements45(  # 计算并保存当前步骤的中间状态。
    different_logits45, original_edge45, selected_edge45, seed=1  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
different_labels45, _ = rtd_labels45(  # 计算并保存当前步骤的中间状态。
    original_edge45, different_corrupted45,  # 执行当前语句以推进本节示例。
    torch.ones_like(original_edge45, dtype=torch.bool),  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert different_corrupted45[0, 1].item() == 9  # 用受控断言验证关键不变量。
assert different_labels45[0, 1].item() == 1  # 用受控断言验证关键不变量。
assert same_labels45[0, 0].item() == -100 and same_labels45[0, 3].item() == -100  # 用受控断言验证关键不变量。
assert same_eligible45.tolist() == [[False, True, True, False]]  # 用受控断言验证关键不变量。

## 6. Generator CE、Discriminator BCE 与联合权重

generator loss 只平均 selected token；discriminator loss 平均所有普通、非 padding token。两者监督密度差异很大，因此联合目标显式写成 generator_loss + lambda·discriminator_loss。

lambda 是优化超参数，不应把两个未归一化 loss 直接相加。下面用小张量逐项重算，防止 ignore mask 或平均分母写错。

In [ ]:
def generator_loss45(logits, mlm_labels):  # 定义本节可复用的核心函数。
    supervised = mlm_labels.ne(-100)  # 计算并保存当前步骤的中间状态。
    if logits.shape[:2] != mlm_labels.shape or not bool(supervised.any()):  # 按当前条件选择后续控制路径。
        raise ValueError("generator loss 缺少合法监督")  # 遇到非法合同立即显式失败。
    return F.cross_entropy(  # 返回当前分支计算出的结果。
        logits.reshape(-1, logits.shape[-1]), mlm_labels.reshape(-1), ignore_index=-100  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。


def discriminator_loss45(logits, labels):  # 定义本节可复用的核心函数。
    supervised = labels.ne(-100)  # 计算并保存当前步骤的中间状态。
    if logits.shape != labels.shape or not bool(supervised.any()):  # 按当前条件选择后续控制路径。
        raise ValueError("discriminator loss 缺少合法监督")  # 遇到非法合同立即显式失败。
    return F.binary_cross_entropy_with_logits(  # 返回当前分支计算出的结果。
        logits[supervised], labels[supervised].float()  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。


def electra_objective45(  # 定义本节可复用的核心函数。
    model, original_ids, attention_mask, probability, seed,  # 执行当前语句以推进本节示例。
    discriminator_weight=5.0,  # 计算并保存当前步骤的中间状态。
):  # 执行当前语句以推进本节示例。
    generator_input_ids, mlm_labels, selected, policy = choose_mlm45(  # 计算并保存当前步骤的中间状态。
        original_ids, attention_mask, probability, seed  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    generator_logits = model.generator_logits(generator_input_ids, attention_mask)  # 计算并保存当前步骤的中间状态。
    corrupted_ids = sample_replacements45(  # 计算并保存当前步骤的中间状态。
        generator_logits.detach(), original_ids, selected, seed + REPLACEMENT_SEED_OFFSET45  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    labels, eligible = rtd_labels45(original_ids, corrupted_ids, attention_mask)  # 计算并保存当前步骤的中间状态。
    discriminator_logits = model.discriminator_logits(corrupted_ids, attention_mask)  # 计算并保存当前步骤的中间状态。
    generator_loss = generator_loss45(generator_logits, mlm_labels)  # 计算并保存当前步骤的中间状态。
    discriminator_loss = discriminator_loss45(discriminator_logits, labels)  # 计算并保存当前步骤的中间状态。
    total = generator_loss + discriminator_weight * discriminator_loss  # 计算并保存当前步骤的中间状态。
    return {  # 返回当前分支计算出的结果。
        "total": total, "generator_loss": generator_loss,  # 执行当前语句以推进本节示例。
        "discriminator_loss": discriminator_loss,  # 执行当前语句以推进本节示例。
        "generator_input_ids": generator_input_ids, "corrupted_ids": corrupted_ids,  # 执行当前语句以推进本节示例。
        "mlm_labels": mlm_labels, "rtd_labels": labels,  # 执行当前语句以推进本节示例。
        "selected": selected, "policy": policy, "eligible": eligible,  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。


tiny_generator_logits45 = torch.tensor([[[2.0, 0.0], [0.0, 1.0], [8.0, -8.0]]])  # 计算并保存当前步骤的中间状态。
tiny_generator_labels45 = torch.tensor([[0, 1, -100]])  # 计算并保存当前步骤的中间状态。
actual_generator_loss45 = generator_loss45(tiny_generator_logits45, tiny_generator_labels45)  # 计算并保存当前步骤的中间状态。
manual_generator_loss45 = (  # 计算并保存当前步骤的中间状态。
    -F.log_softmax(tiny_generator_logits45[0, 0], -1)[0]  # 执行当前语句以推进本节示例。
    -F.log_softmax(tiny_generator_logits45[0, 1], -1)[1]  # 执行当前语句以推进本节示例。
) / 2  # 执行当前语句以推进本节示例。
assert torch.allclose(actual_generator_loss45, manual_generator_loss45, atol=1e-7)  # 用受控断言验证关键不变量。

tiny_discriminator_logits45 = torch.tensor([[0.0, math.log(3.0), -9.0]])  # 计算并保存当前步骤的中间状态。
tiny_discriminator_labels45 = torch.tensor([[0, 1, -100]])  # 计算并保存当前步骤的中间状态。
actual_discriminator_loss45 = discriminator_loss45(  # 计算并保存当前步骤的中间状态。
    tiny_discriminator_logits45, tiny_discriminator_labels45  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
manual_discriminator_loss45 = (  # 计算并保存当前步骤的中间状态。
    F.softplus(tiny_discriminator_logits45[0, 0])  # 执行当前语句以推进本节示例。
    + F.softplus(-tiny_discriminator_logits45[0, 1])  # 执行当前语句以推进本节示例。
) / 2  # 执行当前语句以推进本节示例。
assert torch.allclose(actual_discriminator_loss45, manual_discriminator_loss45, atol=1e-7)  # 用受控断言验证关键不变量。

objective_probe45 = electra_objective45(  # 计算并保存当前步骤的中间状态。
    model45, train_ids45, train_attention45, 0.4, SEED45  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
assert torch.allclose(  # 用受控断言验证关键不变量。
    objective_probe45["total"],  # 执行当前语句以推进本节示例。
    objective_probe45["generator_loss"] + 5.0 * objective_probe45["discriminator_loss"],  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
assert objective_probe45["rtd_labels"][~objective_probe45["eligible"]].eq(-100).all()  # 用受控断言验证关键不变量。
assert objective_probe45["corrupted_ids"][~objective_probe45["selected"]].equal(  # 用受控断言验证关键不变量。
    train_ids45[~objective_probe45["selected"]]  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。

## 7. 受控联合预训练

每一步改变局部 seed，让模型看到不同 MLM 位置与采样结果；不使用验证记录更新参数。最终在一组固定 seed 上比较 generator loss，并检查梯度有限。RTD 的正例比例会随 generator 变强而变化，所以 discriminator accuracy 不能脱离类别比例直接解释。

In [ ]:
torch.manual_seed(SEED45)  # 执行当前语句以推进本节示例。
model45 = ElectraPretrainer45(  # 计算并保存当前步骤的中间状态。
    len(VOCAB45), dim=16, heads=4, hidden_dim=32,  # 计算并保存当前步骤的中间状态。
    generator_layers=1, discriminator_layers=2, max_length=16,  # 计算并保存当前步骤的中间状态。
    share_embeddings=True,  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
optimizer45 = torch.optim.Adam(model45.parameters(), lr=0.015)  # 计算并保存当前步骤的中间状态。

def evaluate_generator45(model, seeds):  # 定义本节可复用的核心函数。
    values = []  # 计算并保存当前步骤的中间状态。
    model.eval()  # 执行当前语句以推进本节示例。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        for seed45 in seeds:  # 遍历输入元素以累积或检查结果。
            result45 = electra_objective45(  # 计算并保存当前步骤的中间状态。
                model, train_ids45, train_attention45, 0.4, seed45  # 执行当前语句以推进本节示例。
            )  # 执行当前语句以推进本节示例。
            values.append(float(result45["generator_loss"]))  # 执行当前语句以推进本节示例。
    return sum(values) / len(values)  # 返回当前分支计算出的结果。


evaluation_seeds45 = [SEED45 + offset for offset in range(4)]  # 计算并保存当前步骤的中间状态。
initial_generator_loss45 = evaluate_generator45(model45, evaluation_seeds45)  # 计算并保存当前步骤的中间状态。
history45 = []  # 计算并保存当前步骤的中间状态。
model45.train()  # 执行当前语句以推进本节示例。
for step in range(45):  # 遍历输入元素以累积或检查结果。
    optimizer45.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    result45 = electra_objective45(  # 计算并保存当前步骤的中间状态。
        model45, train_ids45, train_attention45,  # 执行当前语句以推进本节示例。
        probability=0.4, seed=SEED45 + (step % 12),  # 计算并保存当前步骤的中间状态。
        discriminator_weight=5.0,  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。
    result45["total"].backward()  # 执行当前语句以推进本节示例。
    torch.nn.utils.clip_grad_norm_(model45.parameters(), 1.0)  # 执行当前语句以推进本节示例。
    optimizer45.step()  # 执行当前语句以推进本节示例。
    if step in {0, 9, 29, 44}:  # 按当前条件选择后续控制路径。
        history45.append((  # 执行当前语句以推进本节示例。
            step, float(result45["generator_loss"].detach()),  # 执行当前语句以推进本节示例。
            float(result45["discriminator_loss"].detach()),  # 执行当前语句以推进本节示例。
        ))  # 执行当前语句以推进本节示例。

final_generator_loss45 = evaluate_generator45(model45, evaluation_seeds45)  # 计算并保存当前步骤的中间状态。
model45.eval()  # 执行当前语句以推进本节示例。
fixed_result45 = electra_objective45(  # 计算并保存当前步骤的中间状态。
    model45, train_ids45, train_attention45, 0.4, SEED45  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    valid_records45 = [record for record in RAW_RECORDS45 if record["id"] in VALID_IDS45]  # 计算并保存当前步骤的中间状态。
    valid_ids45, valid_attention45 = collate_records45(valid_records45)  # 计算并保存当前步骤的中间状态。
    valid_result45 = electra_objective45(  # 计算并保存当前步骤的中间状态。
        model45, valid_ids45, valid_attention45, 0.4, SEED45 + 99  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。

assert math.isfinite(initial_generator_loss45) and math.isfinite(final_generator_loss45)  # 用受控断言验证关键不变量。
assert final_generator_loss45 < initial_generator_loss45 * 0.55  # 用受控断言验证关键不变量。
assert torch.isfinite(fixed_result45["total"])  # 用受控断言验证关键不变量。
assert torch.isfinite(valid_result45["total"])  # 用受控断言验证关键不变量。
assert fixed_result45["eligible"].sum().item() == 4 * 5  # 用受控断言验证关键不变量。
assert valid_result45["eligible"].sum().item() == 2 * 5  # 用受控断言验证关键不变量。
positive_rate45 = fixed_result45["rtd_labels"][fixed_result45["eligible"]].float().mean().item()  # 计算并保存当前步骤的中间状态。
assert 0.0 <= positive_rate45 <= 1.0  # 用受控断言验证关键不变量。
print({  # 执行当前语句以推进本节示例。
    "initial_generator_loss": round(initial_generator_loss45, 4),  # 执行当前语句以推进本节示例。
    "final_generator_loss": round(final_generator_loss45, 4),  # 执行当前语句以推进本节示例。
    "rtd_positive_rate": round(positive_rate45, 3),  # 执行当前语句以推进本节示例。
    "trace": history45,  # 执行当前语句以推进本节示例。
})  # 执行当前语句以推进本节示例。

## 8. 双向性、padding 不变性与梯度归属

ELECTRA encoder 是双向的：改变一个未来普通 token，左侧表示通常应变化；但追加 masked padding 不应改变原序列表示。共享 embedding 必须同时从 generator CE 和 discriminator BCE 收到梯度。

这些结构 oracle 与任务指标互补：前者检查实现合同，后者才回答模型是否在目标分布有效。

联合 backward 不能证明两条目标各自连通：其中一条断梯度时，另一条仍可能让共享 embedding 的总梯度非零。因此下面分别只反传 generator CE 与 discriminator BCE，既检查共享 embedding 的非零梯度，也检查非对应的 discriminator/generator 专属 head 保持 `grad is None`。


In [ ]:
model45.eval()  # 执行当前语句以推进本节示例。
bidirectional_a45 = torch.tensor([[CLS45, 5, 9, 7, SEP45]])  # 计算并保存当前步骤的中间状态。
bidirectional_b45 = bidirectional_a45.clone()  # 计算并保存当前步骤的中间状态。
bidirectional_b45[0, 3] = 18  # 计算并保存当前步骤的中间状态。
bidirectional_mask45 = torch.ones_like(bidirectional_a45, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    hidden_a45 = model45.discriminator(bidirectional_a45, bidirectional_mask45)  # 计算并保存当前步骤的中间状态。
    hidden_b45 = model45.discriminator(bidirectional_b45, bidirectional_mask45)  # 计算并保存当前步骤的中间状态。
assert not torch.allclose(hidden_a45[:, 1], hidden_b45[:, 1])  # 用受控断言验证关键不变量。

extended_ids45 = torch.cat([bidirectional_a45, torch.tensor([[18, 19]])], dim=1)  # 计算并保存当前步骤的中间状态。
extended_mask45 = torch.cat(  # 计算并保存当前步骤的中间状态。
    [bidirectional_mask45, torch.zeros(1, 2, dtype=torch.bool)], dim=1  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    hidden_extended45 = model45.discriminator(extended_ids45, extended_mask45)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(hidden_a45, hidden_extended45[:, :5], atol=2e-5)  # 用受控断言验证关键不变量。
assert hidden_extended45[:, 5:].abs().max().item() == 0.0  # 用受控断言验证关键不变量。

model45.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
gradient_result45 = electra_objective45(  # 计算并保存当前步骤的中间状态。
    model45, train_ids45, train_attention45, 0.4, SEED45 + 123  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
gradient_result45["total"].backward()  # 执行当前语句以推进本节示例。
gradients45 = [parameter.grad for parameter in model45.parameters() if parameter.grad is not None]  # 计算并保存当前步骤的中间状态。
assert gradients45 and all(torch.isfinite(gradient).all() for gradient in gradients45)  # 用受控断言验证关键不变量。
shared_gradient45 = model45.generator.token_embedding.weight.grad  # 计算并保存当前步骤的中间状态。
assert shared_gradient45 is model45.discriminator.token_embedding.weight.grad  # 用受控断言验证关键不变量。
assert shared_gradient45.abs().sum().item() > 0  # 用受控断言验证关键不变量。
assert model45.discriminator_head.weight.grad.abs().sum().item() > 0  # 用受控断言验证关键不变量。

# 两条 loss 分开反传：共享 embedding 必须分别收到梯度，非对应 head 不得收到梯度。
model45.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
generator_only_logits45 = model45.generator_logits(generator_input45, train_attention45)  # 计算并保存当前步骤的中间状态。
generator_loss45(generator_only_logits45, mlm_labels45).backward()  # 执行当前语句以推进本节示例。
generator_shared_grad45 = model45.generator.token_embedding.weight.grad  # 计算并保存当前步骤的中间状态。
assert generator_shared_grad45 is not None and generator_shared_grad45.abs().sum().item() > 0  # 用受控断言验证关键不变量。
assert model45.discriminator_head.weight.grad is None  # 用受控断言验证关键不变量。
assert model45.discriminator.blocks[0].ff1.weight.grad is None  # 用受控断言验证关键不变量。

model45.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    discriminator_generator_logits45 = model45.generator_logits(generator_input45, train_attention45)  # 计算并保存当前步骤的中间状态。
    discriminator_corrupted45 = sample_replacements45(  # 计算并保存当前步骤的中间状态。
        discriminator_generator_logits45, train_ids45, selected45,  # 执行当前语句以推进本节示例。
        SEED45 + REPLACEMENT_SEED_OFFSET45,  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    discriminator_labels45, _ = rtd_labels45(  # 计算并保存当前步骤的中间状态。
        train_ids45, discriminator_corrupted45, train_attention45,  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
discriminator_only_logits45 = model45.discriminator_logits(discriminator_corrupted45, train_attention45)  # 计算并保存当前步骤的中间状态。
discriminator_loss45(discriminator_only_logits45, discriminator_labels45).backward()  # 执行当前语句以推进本节示例。
discriminator_shared_grad45 = model45.generator.token_embedding.weight.grad  # 计算并保存当前步骤的中间状态。
assert discriminator_shared_grad45 is not None and discriminator_shared_grad45.abs().sum().item() > 0  # 用受控断言验证关键不变量。
assert model45.generator_bias.grad is None  # 用受控断言验证关键不变量。
assert model45.generator.blocks[0].ff1.weight.grad is None  # 用受控断言验证关键不变量。


# 两条 loss 分开反传：共享 embedding 必须分别收到梯度，非对应 head 不得收到梯度。
model45.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
generator_only_logits45 = model45.generator_logits(generator_input45, train_attention45)  # 计算并保存当前步骤的中间状态。
generator_loss45(generator_only_logits45, mlm_labels45).backward()  # 执行当前语句以推进本节示例。
generator_shared_grad45 = model45.generator.token_embedding.weight.grad  # 计算并保存当前步骤的中间状态。
assert generator_shared_grad45 is not None and generator_shared_grad45.abs().sum().item() > 0  # 用受控断言验证关键不变量。
assert model45.discriminator_head.weight.grad is None  # 用受控断言验证关键不变量。
assert model45.discriminator.blocks[0].ff1.weight.grad is None  # 用受控断言验证关键不变量。

model45.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    discriminator_generator_logits45 = model45.generator_logits(generator_input45, train_attention45)  # 计算并保存当前步骤的中间状态。
    discriminator_corrupted45 = sample_replacements45(  # 计算并保存当前步骤的中间状态。
        discriminator_generator_logits45, train_ids45, selected45,  # 执行当前语句以推进本节示例。
        SEED45 + REPLACEMENT_SEED_OFFSET45,  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    discriminator_labels45, _ = rtd_labels45(  # 计算并保存当前步骤的中间状态。
        train_ids45, discriminator_corrupted45, train_attention45,  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
discriminator_only_logits45 = model45.discriminator_logits(discriminator_corrupted45, train_attention45)  # 计算并保存当前步骤的中间状态。
discriminator_loss45(discriminator_only_logits45, discriminator_labels45).backward()  # 执行当前语句以推进本节示例。
discriminator_shared_grad45 = model45.generator.token_embedding.weight.grad  # 计算并保存当前步骤的中间状态。
assert discriminator_shared_grad45 is not None and discriminator_shared_grad45.abs().sum().item() > 0  # 用受控断言验证关键不变量。
assert model45.generator_bias.grad is None  # 用受控断言验证关键不变量。
assert model45.generator.blocks[0].ff1.weight.grad is None  # 用受控断言验证关键不变量。


## 9. 可信发布：完整绑定 replacement 语义

manifest 不只保存模型尺寸，还绑定完整词表、原始记录、互斥 split、special 排除规则、mask 概率、采样温度、真假标签公式、loss 权重和训练 recipe。否则相同权重配上不同 tokenizer 或把 selected 当 replaced，都可能静默改变模型语义。

canonical state digest 包含 key、dtype、shape、bytes；外部只读 publisher registry 保存整个 package 的预期指纹。包内 hash 可做完整性提示，但攻击者整体替换并重算它们仍必须被 registry 拒绝。

In [ ]:
def canonical_json45(value):  # 定义本节可复用的核心函数。
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")  # 返回当前分支计算出的结果。


def canonical_state_digest45(state):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        tensor = state[key].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        digest.update(canonical_json45({  # 执行当前语句以推进本节示例。
            "key": key, "dtype": str(tensor.dtype), "shape": list(tensor.shape)  # 执行当前语句以推进本节示例。
        }))  # 执行当前语句以推进本节示例。
        digest.update(tensor.numpy().tobytes(order="C"))  # 计算并保存当前步骤的中间状态。
    return digest.hexdigest()  # 返回当前分支计算出的结果。


def package_fingerprint45(package):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    digest.update(canonical_json45(package["manifest"]))  # 执行当前语句以推进本节示例。
    digest.update(canonical_state_digest45(package["state"]).encode("ascii"))  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。


manifest45 = {  # 计算并保存当前步骤的中间状态。
    "subject": "electra-pretraining-demo@1",  # 执行当前语句以推进本节示例。
    "architecture": copy.deepcopy(model45.config),  # 执行当前语句以推进本节示例。
    "vocab": list(VOCAB45),  # 执行当前语句以推进本节示例。
    "special_ids": {  # 执行当前语句以推进本节示例。
        "pad": PAD45, "cls": CLS45, "sep": SEP45,  # 执行当前语句以推进本节示例。
        "mask": MASK45, "unk": UNK45,  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
    "dataset": copy.deepcopy(RAW_RECORDS45),  # 执行当前语句以推进本节示例。
    "split": {"train": list(TRAIN_IDS45), "validation": list(VALID_IDS45)},  # 执行当前语句以推进本节示例。
    "preprocess": {  # 执行当前语句以推进本节示例。
        "tokenizer": "frozen-token-id-v1", "padding": "right",  # 执行当前语句以推进本节示例。
        "mlm_probability": 0.4, "replacement_temperature": 1.0,  # 执行当前语句以推进本节示例。
        "generator_input_policy": {"mask": 0.8, "random_ordinary": 0.1, "unchanged": 0.1},  # 执行当前语句以推进本节示例。
        "selection_count": "round(candidate_count*probability)-minimum-one-per-nonempty-row",  # 执行当前语句以推进本节示例。
        "seed_derivation": {  # 执行当前语句以推进本节示例。
            "selection": 0, "policy": POLICY_SEED_OFFSET45,  # 执行当前语句以推进本节示例。
            "random_token": RANDOM_TOKEN_SEED_OFFSET45,  # 执行当前语句以推进本节示例。
            "generator_replacement": REPLACEMENT_SEED_OFFSET45,  # 执行当前语句以推进本节示例。
        },  # 执行当前语句以推进本节示例。
        "sample_domain": list(ORDINARY45),  # 执行当前语句以推进本节示例。
        "rtd_truth": "corrupted_id != original_id",  # 计算并保存当前步骤的中间状态。
        "excluded_from_losses": sorted(SPECIAL45),  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
    "recipe": {  # 执行当前语句以推进本节示例。
        "seed": SEED45, "optimizer": "Adam", "learning_rate": 0.015,  # 执行当前语句以推进本节示例。
        "steps": 45, "gradient_clip": 1.0,  # 执行当前语句以推进本节示例。
        "generator_loss": "selected-token-cross-entropy",  # 执行当前语句以推进本节示例。
        "discriminator_loss": "eligible-token-bce-with-logits",  # 执行当前语句以推进本节示例。
        "discriminator_weight": 5.0,  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
state45 = {key: value.detach().cpu().clone() for key, value in model45.state_dict().items()}  # 计算并保存当前步骤的中间状态。
package45 = {  # 计算并保存当前步骤的中间状态。
    "manifest": manifest45,  # 执行当前语句以推进本节示例。
    "state": state45,  # 执行当前语句以推进本节示例。
    "internal": {  # 执行当前语句以推进本节示例。
        "manifest_digest": hashlib.sha256(canonical_json45(manifest45)).hexdigest(),  # 执行当前语句以推进本节示例。
        "state_digest": canonical_state_digest45(state45),  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
subject45 = manifest45["subject"]  # 计算并保存当前步骤的中间状态。
PUBLISHER_REGISTRY45 = MappingProxyType({subject45: package_fingerprint45(package45)})  # 计算并保存当前步骤的中间状态。


def load_published_electra45(package, subject):  # 定义本节可复用的核心函数。
    if subject not in PUBLISHER_REGISTRY45:  # 按当前条件选择后续控制路径。
        raise ValueError("未知发布 subject")  # 遇到非法合同立即显式失败。
    if package_fingerprint45(package) != PUBLISHER_REGISTRY45[subject]:  # 按当前条件选择后续控制路径。
        raise ValueError("publisher registry 指纹不匹配")  # 遇到非法合同立即显式失败。
    manifest = package["manifest"]  # 计算并保存当前步骤的中间状态。
    if manifest["subject"] != subject:  # 按当前条件选择后续控制路径。
        raise ValueError("subject 不匹配")  # 遇到非法合同立即显式失败。
    if package["internal"]["manifest_digest"] != hashlib.sha256(canonical_json45(manifest)).hexdigest():  # 按当前条件选择后续控制路径。
        raise ValueError("manifest 内部摘要不匹配")  # 遇到非法合同立即显式失败。
    if package["internal"]["state_digest"] != canonical_state_digest45(package["state"]):  # 按当前条件选择后续控制路径。
        raise ValueError("state 内部摘要不匹配")  # 遇到非法合同立即显式失败。
    if manifest["vocab"] != VOCAB45 or manifest["special_ids"]["mask"] != MASK45:  # 按当前条件选择后续控制路径。
        raise ValueError("词表或 special id 不匹配")  # 遇到非法合同立即显式失败。
    train_ids = set(manifest["split"]["train"])  # 计算并保存当前步骤的中间状态。
    validation_ids = set(manifest["split"]["validation"])  # 计算并保存当前步骤的中间状态。
    data_ids = {record["id"] for record in manifest["dataset"]}  # 计算并保存当前步骤的中间状态。
    if train_ids & validation_ids or train_ids | validation_ids != data_ids:  # 按当前条件选择后续控制路径。
        raise ValueError("split 非互斥或未覆盖")  # 遇到非法合同立即显式失败。
    expected_policy45 = {"mask": 0.8, "random_ordinary": 0.1, "unchanged": 0.1}  # 计算并保存当前步骤的中间状态。
    expected_seed_derivation45 = {  # 计算并保存当前步骤的中间状态。
        "selection": 0, "policy": POLICY_SEED_OFFSET45,  # 执行当前语句以推进本节示例。
        "random_token": RANDOM_TOKEN_SEED_OFFSET45,  # 执行当前语句以推进本节示例。
        "generator_replacement": REPLACEMENT_SEED_OFFSET45,  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    if manifest["preprocess"]["generator_input_policy"] != expected_policy45:  # 按当前条件选择后续控制路径。
        raise ValueError("generator 80/10/10 策略不匹配")  # 遇到非法合同立即显式失败。
    if manifest["preprocess"]["selection_count"] != "round(candidate_count*probability)-minimum-one-per-nonempty-row":  # 按当前条件选择后续控制路径。
        raise ValueError("MLM selection/min-one 策略不匹配")  # 遇到非法合同立即显式失败。
    if manifest["preprocess"]["seed_derivation"] != expected_seed_derivation45:  # 按当前条件选择后续控制路径。
        raise ValueError("随机流派生规则不匹配")  # 遇到非法合同立即显式失败。
    expected_policy45 = {"mask": 0.8, "random_ordinary": 0.1, "unchanged": 0.1}  # 计算并保存当前步骤的中间状态。
    expected_seed_derivation45 = {  # 计算并保存当前步骤的中间状态。
        "selection": 0, "policy": POLICY_SEED_OFFSET45,  # 执行当前语句以推进本节示例。
        "random_token": RANDOM_TOKEN_SEED_OFFSET45,  # 执行当前语句以推进本节示例。
        "generator_replacement": REPLACEMENT_SEED_OFFSET45,  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    if manifest["preprocess"]["generator_input_policy"] != expected_policy45:  # 按当前条件选择后续控制路径。
        raise ValueError("generator 80/10/10 策略不匹配")  # 遇到非法合同立即显式失败。
    if manifest["preprocess"]["selection_count"] != "round(candidate_count*probability)-minimum-one-per-nonempty-row":  # 按当前条件选择后续控制路径。
        raise ValueError("MLM selection/min-one 策略不匹配")  # 遇到非法合同立即显式失败。
    if manifest["preprocess"]["seed_derivation"] != expected_seed_derivation45:  # 按当前条件选择后续控制路径。
        raise ValueError("随机流派生规则不匹配")  # 遇到非法合同立即显式失败。
    if manifest["preprocess"]["sample_domain"] != ORDINARY45:  # 按当前条件选择后续控制路径。
        raise ValueError("replacement 采样域不匹配")  # 遇到非法合同立即显式失败。
    if manifest["preprocess"]["rtd_truth"] != "corrupted_id != original_id":  # 按当前条件选择后续控制路径。
        raise ValueError("RTD 标签语义不匹配")  # 遇到非法合同立即显式失败。
    if manifest["preprocess"]["excluded_from_losses"] != sorted(SPECIAL45):  # 按当前条件选择后续控制路径。
        raise ValueError("special 排除规则不匹配")  # 遇到非法合同立即显式失败。
    for record in manifest["dataset"]:  # 遍历输入元素以累积或检查结果。
        if any(token not in ORDINARY45 for token in record["tokens"]):  # 按当前条件选择后续控制路径。
            raise ValueError("原始数据含非法 token")  # 遇到非法合同立即显式失败。
    loaded = ElectraPretrainer45(**manifest["architecture"])  # 计算并保存当前步骤的中间状态。
    loaded.load_state_dict(package["state"], strict=True)  # 计算并保存当前步骤的中间状态。
    loaded.eval()  # 执行当前语句以推进本节示例。
    return loaded  # 返回当前分支计算出的结果。


loaded45 = load_published_electra45(package45, subject45)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    assert torch.allclose(  # 用受控断言验证关键不变量。
        loaded45.generator_logits(generator_input45, train_attention45),  # 执行当前语句以推进本节示例。
        model45.generator_logits(generator_input45, train_attention45),  # 执行当前语句以推进本节示例。
        atol=1e-7,  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。
    assert torch.allclose(  # 用受控断言验证关键不变量。
        loaded45.discriminator_logits(train_ids45, train_attention45),  # 执行当前语句以推进本节示例。
        model45.discriminator_logits(train_ids45, train_attention45),  # 执行当前语句以推进本节示例。
        atol=1e-7,  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。
assert canonical_state_digest45(state45) == package45["internal"]["state_digest"]  # 用受控断言验证关键不变量。

forged45 = copy.deepcopy(package45)  # 计算并保存当前步骤的中间状态。
key45 = sorted(forged45["state"])[0]  # 计算并保存当前步骤的中间状态。
forged45["state"][key45].view(-1)[0] += 1.0  # 计算并保存当前步骤的中间状态。
forged45["internal"]["state_digest"] = canonical_state_digest45(forged45["state"])  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    load_published_electra45(forged45, subject45)  # 执行当前语句以推进本节示例。
    raise AssertionError("重算内部 state hash 的伪造未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as error45:  # 捕获预期异常并验证失败分支。
    assert "registry" in str(error45)  # 用受控断言验证关键不变量。

replacement45 = copy.deepcopy(package45)  # 计算并保存当前步骤的中间状态。
replacement45["manifest"]["preprocess"]["rtd_truth"] = "selected_position"  # 计算并保存当前步骤的中间状态。
replacement_key45 = sorted(replacement45["state"])[-1]  # 计算并保存当前步骤的中间状态。
replacement45["state"][replacement_key45].view(-1)[-1] -= 0.25  # 计算并保存当前步骤的中间状态。
replacement45["internal"]["manifest_digest"] = hashlib.sha256(  # 计算并保存当前步骤的中间状态。
    canonical_json45(replacement45["manifest"])  # 执行当前语句以推进本节示例。
).hexdigest()  # 执行当前语句以推进本节示例。
replacement45["internal"]["state_digest"] = canonical_state_digest45(  # 计算并保存当前步骤的中间状态。
    replacement45["state"]  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    load_published_electra45(replacement45, subject45)  # 执行当前语句以推进本节示例。
    raise AssertionError("整体替换并重算内部 hash 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as error45:  # 捕获预期异常并验证失败分支。
    assert "registry" in str(error45)  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    PUBLISHER_REGISTRY45[subject45] = "attacker"  # 计算并保存当前步骤的中间状态。
    raise AssertionError("registry 不应可写")  # 遇到非法合同立即显式失败。
except TypeError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

## 10. 失败模式、生产差距与资料

高频错误包括：把 selected 位置直接标成 replaced；允许采样 MASK/CLS；对 padding 算 BCE；generator logits 未 detach 就让 RTD 反向穿过离散采样；embedding 名义共享但实际复制；切分发生在 masking 之后；只报 discriminator accuracy 却忽略严重类别不平衡。

生产预训练还需要大规模 tokenizer 与动态 masking、数据去重和污染审计、generator/discriminator 容量搜索、分布式混合精度、checkpoint 恢复、吞吐与显存分析。下游价值必须在冻结评测协议和独立数据上验证。

原始资料：

- [ELECTRA: Pre-training Text Encoders as Discriminators Rather Than Generators](https://arxiv.org/abs/2003.10555)
- [Google Research ELECTRA 官方实现](https://github.com/google-research/electra)
- [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805)